<a href="https://colab.research.google.com/github/aishanikar9/BWSI_Operations_Team/blob/main/SegmentationModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kagglehub
import kagglehub, pathlib, numpy as np
import matplotlib.pyplot as plt
from PIL import Image

path = kagglehub.dataset_download("yaroslavchyrko/rescuenet")
print("downloaded to:", path)

100%|██████████| 21.9G/21.9G [04:33<00:00, 85.8MB/s]

Extracting files...


downloaded to: /root/.cache/kagglehub/datasets/yaroslavchyrko/rescuenet/versions/1


In [ ]:
root = pathlib.Path(path)
org_dir   = list(root.rglob("train-org-img"))[0]
label_dir = list(root.rglob("train-label-img"))[0]
print("images:", len(list(org_dir.glob("*.jpg"))), "| masks:", len(list(label_dir.glob("*.png"))))

images: 3595 | masks: 3595


In [ ]:
#0=Unlabeled, 1= Water, 2 = Building w/o Damage
#3 Building with minor damage, 4 Building with Major damage
#5 Building completely destroyed, 6 Vechicle, 7 Clear Road
#8 Blocked Road, 9 Tree, 10 Pool

!pip install -q segmentation-models-pytorch
import segmentation_models_pytorch as smp
import torch, torch.nn as nn, torch.optim as optim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:00


In [ ]:
class_number = 11

model = smp.Unet(encoder_name="resnet34",
                 encoder_weights="imagenet",
                 in_channels=3,
                 classes=class_number,
                 )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
path = kagglehub.dataset_download("yaroslavchyrko/rescuenet")
root = pathlib.Path(path)

train_orginal   = list(root.rglob("train-org-img"))[0]
train_label = list(root.rglob("train-label-img"))[0]
val_orginal     = list(root.rglob("val-org-img"))[0]
val_label   = list(root.rglob("val-label-img"))[0]
print("train imgs/masks:", len(list(train_orginal.glob('*.jpg'))), len(list(train_label.glob('*.png'))))
print("val imgs/masks:  ", len(list(val_orginal.glob('*.jpg'))), len(list(val_label.glob('*.png'))))

Using Colab cache for faster access to the 'rescuenet' dataset.
train imgs/masks: 3595 3595
val imgs/masks:   449 449


In [ ]:
loss = nn.CrossEntropyLoss()
dice_loss = smp.losses.DiceLoss(mode="multiclass") # diceloss is used common in image segmentation by focusing on the intersection of predicted mask and the real mask

def criterion(logits, masks):
  return loss(logits, masks) + dice_loss(logits, masks)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, 0.9)

In [ ]:
import pathlib, numpy as np
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

In [ ]:
#RescueNetSegmentedDataset class, Kento

In [ ]:
#build the data sets from org_dir, label_dir, see ResNetModel for reference, Oluj

In [ ]:
#IoU or mIOU evaluation function, Aishani
def calc_IOU(pred_arr, label_arr):
  #multiclass - each class has separate IOU score?
  iou_scores = []
  #union pix
  for c_index in range(class_number):
    #get where they overlap and are correct
    class_gt = label_arr == c_index
    class_pred_area = pred_arr == c_index
    intersect_pix = np.sum((class_gt & class_pred_area))
    intersect_pix = float(intersect_pix)

    #get total area of masks merged

    total_region = np.sum((class_gt | class_pred_area))
    total_region = float(total_region)

    if(total_region == 0):
      continue

    iou_scores.append(intersect_pix/total_region)

  return iou_scores

def calc_mean_IOU(iou_scores): # can merge into another output above
  #iou_scores = [score for score in iou_scores if score != 0]
  return np.mean(iou_scores)


#testing
# import PIL
# import os
# from pathlib import Path
# import matplotlib.pyplot as plt
# c = 0
# p_img_masks = []
# for f in Path(train_label).iterdir():
#   if c<5:
#     img = PIL.Image.open(f)
#     img_arr = np.array(img)
#     p_img_masks.append(img_arr)
#   c+=1

# print(calc_IOU(p_img_masks[4],p_img_masks[1]))
# fix ,ax = plt.subplots(1,2,figsize=(6,6))

# ax[0].imshow(p_img_masks[4])
# ax[1].imshow(p_img_masks[1])

In [ ]:
#training, saving checkpoints to drive, Brian

In [ ]:
#run it